**Import Libraries**

In [9]:
import torch
import pandas as pd
import numpy as np
import os
from sentence_transformers import SentenceTransformer
import warnings

warnings.filterwarnings("ignore")

**Load the BGE Model**

In [10]:
print("Loading BAAI/bge-base-en-v1.5 model from Hugging Face...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

text_model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)
print("Text model loaded successfully!")

Loading BAAI/bge-base-en-v1.5 model from Hugging Face...
Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text model loaded successfully!


**Process the Transcripts**

In [11]:
df = pd.read_csv("../data/processed/cleaned_transcripts.csv")

texts = df['text'].fillna("").astype(str).tolist()
labels = df['label'].tolist()

print(f"Extracting 768-dim embeddings for {len(texts)} transcripts...")
text_embeddings = text_model.encode(texts, show_progress_bar=True)

print(f"Successfully extracted shape: {text_embeddings.shape}")

Extracting 768-dim embeddings for 800 transcripts...


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Successfully extracted shape: (800, 768)


In [12]:
embedding_dim = text_embeddings.shape[1]

col_names = [f'text_dim_{i}' for i in range(1, embedding_dim + 1)]
text_features_df = pd.DataFrame(text_embeddings, columns=col_names)

text_features_df['text'] = texts
text_features_df['label'] = labels

final_path = "../data/processed/text_features.csv"
text_features_df.to_csv(final_path, index=False)

print(f"Saved text dataset to: {final_path}")
text_features_df.head()

Saved text dataset to: ../data/processed/text_features.csv


,text_dim_1,text_dim_2,text_dim_3,text_dim_4,text_dim_5,text_dim_6,text_dim_7,text_dim_8,text_dim_9,text_dim_10,...,text_dim_761,text_dim_762,text_dim_763,text_dim_764,text_dim_765,text_dim_766,text_dim_767,text_dim_768,text,label
0,-0.017639,-0.012787,0.019595,-0.011021,0.017845,-0.005269,0.001693,0.029141,0.004444,-0.020721,...,-0.002323,-0.062209,0.033399,-0.010720,0.023304,-0.004868,0.019025,-0.023521,"[Greetings], this is [Name]. I finally got my ...",0
1,-0.051521,-0.023332,0.052157,0.004949,0.014232,0.004946,0.007687,0.018442,-0.029052,-0.006738,...,0.012187,-0.089950,0.001615,-0.019768,-0.038382,0.002893,0.048569,0.012661,"[Greetings], this is [Name]. I am hosting a sm...",0
2,0.033028,-0.022054,0.008742,0.025706,0.034990,-0.019323,-0.027322,0.001458,-0.020370,-0.027367,...,0.054734,-0.063119,0.017895,-0.090105,-0.049253,-0.004698,-0.016453,-0.010527,"[Greetings], this is the [Company] contacting...",1
3,0.020345,-0.039090,0.030919,-0.018577,-0.016390,0.029322,0.004527,0.019309,-0.020050,-0.030155,...,-0.027675,-0.071302,0.006894,-0.057554,-0.012103,0.030787,0.003697,0.004880,"[Greetings], this is [Name] from [Company] Ene...",0
4,0.012248,-0.025037,0.017299,0.029655,0.026188,-0.015081,-0.004176,0.019845,-0.015494,-0.018883,...,0.042799,-0.089319,0.006041,-0.059840,-0.031099,0.010321,0.009607,-0.007890,"[Greetings], this is the [Company] reaching o...",1
